In [50]:
# NB1 - Cell 1: Mount Drive
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [51]:
import os
print(os.path.exists("/content/drive"))

True


In [52]:
PROJECT_ROOT = "/content/drive/MyDrive/CLASS_AlignED_MVP"

In [53]:
# NB1 - Cell 2: Install deps
!pip -q install pypdf python-docx

In [54]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [55]:
import os, json, re, hashlib
from pathlib import Path

PROJECT_ROOT = "/content/drive/MyDrive/CLASS_AlignED_MVP"
RAW_SYLLABI = f"{PROJECT_ROOT}/raw/syllabi"
OUT_TEXT = f"{PROJECT_ROOT}/processed/text"
OUT_CHUNKS = f"{PROJECT_ROOT}/processed/chunks"

Path(OUT_TEXT).mkdir(parents=True, exist_ok=True)
Path(OUT_CHUNKS).mkdir(parents=True, exist_ok=True)

def stable_id(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8")).hexdigest()[:12]

In [56]:
# NB1 - Cell 4: Text extraction utilities
from pypdf import PdfReader
from docx import Document

def extract_pdf_text(pdf_path: str) -> str:
    reader = PdfReader(pdf_path)
    pages = []
    for i, p in enumerate(reader.pages):
        t = p.extract_text() or ""
        pages.append(f"\n\n[PAGE {i+1}]\n{t}")
    return "\n".join(pages)

def extract_docx_text(docx_path: str) -> str:
    doc = Document(docx_path)
    return "\n".join([p.text for p in doc.paragraphs if p.text.strip()])

def extract_text(file_path: str) -> str:
    ext = Path(file_path).suffix.lower()
    if ext == ".pdf":
        return extract_pdf_text(file_path)
    if ext == ".docx":
        return extract_docx_text(file_path)
    raise ValueError(f"Unsupported file type: {ext}")

In [57]:
# NB1 - Cell 5: Simple sectioning + chunking
HEADING_RE = re.compile(r"^\s*([A-Z][A-Z0-9 \-/:,&]{6,})\s*$")

def split_into_sections(text: str):
    lines = text.splitlines()
    sections = []
    current_title = "UNSPECIFIED"
    buf = []
    for line in lines:
        if HEADING_RE.match(line.strip()):
            if buf:
                sections.append((current_title, "\n".join(buf).strip()))
                buf = []
            current_title = line.strip()
        else:
            buf.append(line)
    if buf:
        sections.append((current_title, "\n".join(buf).strip()))
    # Drop empty
    return [(t, c) for t, c in sections if c.strip()]

def chunk_section(title: str, content: str, max_chars=1800):
    chunks = []
    content = content.strip()
    if len(content) <= max_chars:
        return [content]
    # paragraph-based splitting
    paras = [p.strip() for p in content.split("\n\n") if p.strip()]
    cur = []
    cur_len = 0
    for p in paras:
        if cur_len + len(p) + 2 > max_chars and cur:
            chunks.append("\n\n".join(cur))
            cur, cur_len = [], 0
        cur.append(p)
        cur_len += len(p) + 2
    if cur:
        chunks.append("\n\n".join(cur))
    return chunks

In [58]:
syllabi_files = sorted([
    str(p) for p in Path(RAW_SYLLABI).glob("*")
    if p.is_file() and p.suffix.lower() in {".pdf", ".docx"}
])

print("Found syllabi:", syllabi_files)

if not syllabi_files:
    raise FileNotFoundError(f"No PDF or DOCX files found in {RAW_SYLLABI}")
manifest = []
for fpath in syllabi_files:
    raw_text = extract_text(fpath)
    base = Path(fpath).stem
    doc_id = stable_id(base + "::" + str(Path(fpath).stat().st_size))

    # Save extracted text JSON
    text_obj = {"doc_id": doc_id, "filename": Path(fpath).name, "text": raw_text}
    text_json_path = f"{OUT_TEXT}/{doc_id}.json"
    with open(text_json_path, "w", encoding="utf-8") as w:
        json.dump(text_obj, w, ensure_ascii=False, indent=2)

    # Chunk into JSONL
    sections = split_into_sections(raw_text)
    chunk_path = f"{OUT_CHUNKS}/{doc_id}.jsonl"
    with open(chunk_path, "w", encoding="utf-8") as w:
        chunk_idx = 0
        for sec_title, sec_content in sections:
            for part in chunk_section(sec_title, sec_content):
                chunk_idx += 1
                chunk_id = stable_id(doc_id + f"::{chunk_idx}::{sec_title}")
                rec = {
                    "doc_id": doc_id,
                    "chunk_id": chunk_id,
                    "source_file": Path(fpath).name,
                    "section": sec_title,
                    "chunk_index": chunk_idx,
                    "text": part
                }
                w.write(json.dumps(rec, ensure_ascii=False) + "\n")

    manifest.append({
        "doc_id": doc_id,
        "source_file": Path(fpath).name,
        "text_json": text_json_path,
        "chunks_jsonl": chunk_path
    })

manifest_path = f"{PROJECT_ROOT}/processed/manifest.json"
with open(manifest_path, "w", encoding="utf-8") as w:
    json.dump(manifest, w, ensure_ascii=False, indent=2)

print("Wrote manifest:", manifest_path)
print("Example record:", manifest[0] if manifest else None)

Found syllabi: ['/content/drive/MyDrive/CLASS_AlignED_MVP/raw/syllabi/MATA 621 Applied Ordinary Dif Equations Spring 2026.pdf', '/content/drive/MyDrive/CLASS_AlignED_MVP/raw/syllabi/Nutrition Syllabus Spring 2026-1.docx', '/content/drive/MyDrive/CLASS_AlignED_MVP/raw/syllabi/Principles of Immunology.pdf']
Wrote manifest: /content/drive/MyDrive/CLASS_AlignED_MVP/processed/manifest.json
Example record: {'doc_id': 'a802c5fba7e7', 'source_file': 'MATA 621 Applied Ordinary Dif Equations Spring 2026.pdf', 'text_json': '/content/drive/MyDrive/CLASS_AlignED_MVP/processed/text/a802c5fba7e7.json', 'chunks_jsonl': '/content/drive/MyDrive/CLASS_AlignED_MVP/processed/chunks/a802c5fba7e7.jsonl'}
